# Microzone Cleanup Notebook

This notebook takes a microzone and traffic analysis zone layer and fills in gaps, then merges them into neighbors without crossing over the taz boundry.

Point workspace variable at the `.gdb` folder for your project

Ensure that microzone and traffic analysis zone are inside of your `.gdb` folder

Point `maz` variable at the name of microzone feature class 

Example: `maz = "microzone_layer"`

Point `taz` variable at the name of traffic analysis zone feature class

Example: `taz = "taz_layer"`




### Import Necessary Libraries

In [13]:
import arcpy

### Assign key Variables

In [14]:
workspace = r"D:\ABM-INP-Microzones\Scripts\Scratch\Microzones_Cleaner\Microzones_Cleaner.gdb"
maz = "microzones_Export"
taz = "TAZ_Export"
mag_maz ="Microzones_COMBINED"

### Point Arcpy at `.gdb` folder
Enable output overwrite

In [36]:
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True

### Clean MAG Data
Copy the combined microzones layer, then keep only UTAH County in `_00_Microzones_Mag`. The `Microzones_COMBINED` source remains unchanged.

In [16]:
mag_maz_clean = "_00_Microzones_Mag"
mag_maz_layer = "mag_maz_clean_lyr"

arcpy.management.CopyFeatures(mag_maz, mag_maz_clean)
arcpy.management.MakeFeatureLayer(mag_maz_clean, mag_maz_layer)
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=mag_maz_layer,
    selection_type="NEW_SELECTION",
    where_clause="CO_NAME = 'UTAH'"
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=mag_maz_layer,
    selection_type="SWITCH_SELECTION"
)
deleted_count = int(arcpy.management.GetCount(mag_maz_layer)[0])
arcpy.management.DeleteRows(mag_maz_layer)

arcpy.management.Delete(mag_maz_layer)
print(f" deleted {deleted_count} non-UTAH rows.")

 deleted 2216 non-UTAH rows.


### Assign Microzone Ids for MAG

Rename the Microzone field to match WFRC convention so it auto joins

In [17]:
arcpy.management.AlterField(
    in_table=mag_maz_clean,
    field="maz_id",
    new_field_name="MAZID",
    new_field_alias = "MAZID"
)

code_block = """
rec = 0
def autoIncrement():
    global rec
    pStart = 11000 
    if (rec == 0): 
        rec = pStart 
    else: 
        rec += 1 
    return rec
"""

arcpy.management.CalculateField(
    in_table=mag_maz_clean,
    field="MAZID",
    expression="autoIncrement()",
    expression_type="PYTHON3",
    code_block=code_block
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_00_Microzones_Mag'>

### Join MAG Microzones to WFRC Microzones

In [18]:
maz_comb = "_01_Microzones_combined"

arcpy.management.Merge([maz,mag_maz_clean],maz_comb)

    

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_01_Microzones_combined'>

### Update Plan Area Field MAG Zones

In [19]:
maz_comb_lyr = "maz_comb_lyr"

arcpy.management.MakeFeatureLayer(maz_comb, maz_comb_lyr)

# 2. Select only the features where CO_NAME is UTAH
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=maz_comb_lyr,
    selection_type="NEW_SELECTION",
    where_clause="CO_NAME = 'UTAH'"
)

# 3. Update PLANAREA to 'MAG' for the selected features
arcpy.management.CalculateField(
    in_table=maz_comb_lyr,
    field="PLANAREA",
    expression="'MAG'",
    expression_type="PYTHON3"
)

<Result 'maz_comb_lyr'>

### Run Identity Tool

This will fill in all gaps between polygons

Creates a new feature class called `maz_i` which has more features than orignial maz becuase these are all new filled in featuers

Original Microzones total = 14,380

MAZID"0" = 863


New Zones = 878

MAZ identified  total = 15,258

In [20]:
maz_i = "_02_maz_i"

arcpy.analysis.Identity(
    in_features=taz, 
    identity_features=maz_comb, 
    out_feature_class=maz_i)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_02_maz_i'>

### Multipart to Singlepart

Identity will treat some new features a single one and this function will break them back into their own

Creates a new feature class called `maz_filled` with all gaps filled in as unique features

MAZ Filled total = 16,194

New Zones = 1,236

MAZID"0" = 1,795

NEW MAZID"0" = 932

Missing duplicate Zones = (new zones [16,194]) - (og zones [14,380]) = 1,814 - (MAZID"0"[1,795]) = 19
The 19 duplicate zones will be filterd out later when I look for sliver zones


In [21]:
maz_filled = "_03_maz_filled"
arcpy.management.MultipartToSinglepart(
    in_features= maz_i,
    out_feature_class=maz_filled)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Create new field for filled gap polygon

All new polygons created from the identify and multipart functions will have `MAZID = 0` so to easily identify them later I will make a dummy column for it
Flags 1,793 polygons


In [22]:
arcpy.management.CalculateField(
    in_table=maz_filled,
    field="gap_flag",
    expression="1 if (!MAZID! == 0 or !MAZID! is None) else 0",
    expression_type="PYTHON3",
    field_type="SHORT"
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Calculate Poslby Posper and Perm/Area Score

In [23]:
pp_code = """
import math
def calc_pp(area, perimeter, current_pp):
    # If it already has a value, keep it (optional), or overwrite if None
    if current_pp is not None:
        return current_pp
    if perimeter == 0 or perimeter is None:
        return 0
    return (4 * math.pi * area) / (perimeter ** 2)
"""

arcpy.management.CalculateField(
    in_table=maz_filled, # Update this variable if you run it later in the script (e.g., maz_sliver)
    field="PolsbyPopper",
    expression="calc_pp(!Shape_Area!, !Shape_Length!, !PolsbyPopper!)",
    expression_type="PYTHON3",
    code_block=pp_code
)

# 2. Calculate Perimeter to Area Ratio
pa_code = """
def calc_pa(area, perimeter, current_pa):
    if current_pa is not None:
        return current_pa
    if area == 0 or area is None:
        return 0
    return perimeter / area
"""

arcpy.management.CalculateField(
    in_table=maz_filled, 
    field="PerimtertoArea",
    expression="calc_pa(!Shape_Area!, !Shape_Length!, !PerimtertoArea!)",
    expression_type="PYTHON3",
    code_block=pa_code
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

## Identify zones for review

In addition to gap zones we want to flag sliver zones

Shape Area is less than 10,000

AND 

PolsbyPopper is less than .03

OR 

PerimtertoArea is less than .03

OR

Shape Area < 1,000


**This will flag 1,995 zones as sliver zones to be merged into the neighboring microzones**

In [24]:
maz_sliver = "_04_maz_sliver"


arcpy.management.CopyFeatures(
    in_features= maz_filled,
    out_feature_class=maz_sliver
)

code_block = """
def check_sliver(pp, pa, area, gap):
    # Guard clause: large features (highways) are never slivers
    if area is not None and area >= 10000:
        return 0
    
    # Check sliver criteria for features under 10,000
    if pp is not None and pp < 0.03:
        return 1
    if pa is not None and pa < 0.03:
        return 1
    if area is not None and area < 1000:
        return 1
    if gap == 1:
        return 1
    return 0
"""

arcpy.management.CalculateField(
    in_table=maz_sliver,
    field="Sliver_Flag",
    expression="check_sliver(!PolsbyPopper!, !PerimtertoArea!, !Shape_Area!, !gap_flag!)",
    expression_type="PYTHON3",
    code_block=code_block,
    field_type="SHORT"
)


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_04_maz_sliver'>

### Additional layer of just sliver zones for eye test

In [25]:
slivers = "_05_iso_slivers"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=slivers
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=slivers, 
    selection_type="NEW_SELECTION", 
    where_clause="Sliver_Flag = 1"
)

arcpy.management.CopyFeatures(
    in_features=slivers,
    out_feature_class=slivers
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_05_iso_slivers'>

### Create Taz vector lines

Eliminate tool requires vector lines to use as a boarder so we dont merge between TAZ

In [26]:
taz_lines = "_06_taz_lines"
arcpy.management.PolygonToLine(taz, taz_lines)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_06_taz_lines'>

### Run Eliminate 

Merges 1,818 features which leaves 177 unmerged features

These 177 are zones that have micro crossover TAZ boundaries



In [27]:
maz_cleaned = "_07_maz_cleaned"
elim_layer = "temp_elim_layer"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=elim_layer
)

arcpy.management.SelectLayerByAttribute(elim_layer, "NEW_SELECTION", 'Sliver_Flag = 1')

arcpy.management.Eliminate(
    in_features=elim_layer, 
    out_feature_class=maz_cleaned,
    ex_features=taz_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_07_maz_cleaned'>

### Snap micro zones to the taz lines to get rid of micro cross over

These zones left over are all small enough to be automatically deleted with Integrate tool

Leaves 14,199 maz left over

In [28]:
maz_snapped = "_08_maz_snapped"

arcpy.management.CopyFeatures(maz_cleaned, maz_snapped)
arcpy.management.Integrate(
    in_features=[[maz_snapped, 1]], 
    cluster_tolerance="1 Meters" # Adjust to "3 Feet" if your projection is in feet
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_08_maz_snapped'>

### Check for microzones that overlap TAZ

2,093 microzones that cross over taz boundaries

In [29]:
snapped_layer = "temp_snapped_layer"
maz_crossover = "_09_maz_crossover"

arcpy.management.MakeFeatureLayer(maz_snapped, snapped_layer)

arcpy.management.SelectLayerByLocation(
    in_layer=snapped_layer,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz
)

arcpy.management.CopyFeatures(snapped_layer, maz_crossover)



<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_09_maz_crossover'>

### Visualize overlapping areas

Shrink all maz by .01 meters to ignore zones that are perfecty ontop of the taz. 

clip all the new maz to taz boundaries

In [30]:
maz_shrunk = "_10_maz_shrunk"
maz_crossover_lines = "_11_maz_crossover_lines"

arcpy.analysis.Buffer(
    in_features=maz_snapped, 
    out_feature_class=maz_shrunk, 
    buffer_distance_or_field="-0.01 Meters"
)


arcpy.analysis.Clip(
    in_features=taz_lines,
    clip_features=maz_shrunk,
    out_feature_class=maz_crossover_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_11_maz_crossover_lines'>

### Split crossover microzones at taz boundaries
### Merge the new cuts with neighboring MAZ
Use TAZ lines to make sure it doesn't merge back over TAZ boundaries

In [31]:
maz_taz_identity = "_12_maz_taz_identity"
maz_cross_2 = "_13_maz_crossing"
temp_elim = "temp_elim"

arcpy.analysis.Identity(
    in_features=maz_snapped,
    identity_features=taz,
    out_feature_class=maz_taz_identity
)

arcpy.management.MakeFeatureLayer(maz_taz_identity, temp_elim)
# 2. Flag the tiny crossover slivers (where the Microzone's parent TAZ doesn't match the physical TAZ it spilled into)
# Assuming your MAZ has a 'TAZ_ID' field and the TAZ layer has a 'TAZ_ID' field

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_elim,
    selection_type="NEW_SELECTION",
    where_clause="TAZID_1 <> TAZID_12"
)

selected_count = int(arcpy.management.GetCount(temp_elim)[0])
print(f"Selected {selected_count} crossover slivers for elimination.")

# 3. Only run Eliminate if features were successfully selected
if selected_count > 0:
    arcpy.management.Eliminate(
        in_features=temp_elim,
        out_feature_class=maz_cross_2,
        ex_features=taz_lines # This correctly prevents them from merging back across the TAZ line
    )
else:
    print("Eliminate skipped: No features selected. Double check your field names.")

Selected 15266 crossover slivers for elimination.



`_12_maz_taz_identiy` has 29,465 features 

Identiy created 15,266 new sliver zones where microzones cross over a taz line.

Eliminate tool merged 13,937 features back into neighboring maz

but the eliminate tool wasn't able to merge all of them into new zones. 

There are 1,329 polygons that didn't merge

 `_08_maz_snapped` count (14,199) - `13_maz_crossing` count(15,528) = 1,329


### Create new layer of just these new sliver zones

In [32]:
stuck_slivers = "_14_stuck_slivers"
temp_stuck = "temp_stuck_layer"

# Create a layer from your Eliminate output
arcpy.management.MakeFeatureLayer(
    in_features=maz_cross_2, 
    out_layer=temp_stuck
)

# Use the exact same WHERE clause from your Eliminate step
# (Make sure to use the correct suffix, e.g., TAZID_1 or TAZID_2)
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_stuck,
    selection_type="NEW_SELECTION",
    where_clause="TAZID_1 <> TAZID_12" 
)

# Export them to their own layer so you can inspect them
arcpy.management.CopyFeatures(
    in_features=temp_stuck, 
    out_feature_class=stuck_slivers
)

# Print the count to confirm
stuck_count = int(arcpy.management.GetCount(temp_stuck)[0])
print(f"Found {stuck_count} orphaned polygons that failed to merge.")

Found 1329 orphaned polygons that failed to merge.


### All orphaned polygons are super small so merge into MAZ

All 1329 are smaller than 50 meters shape area so we can safely merge them within taz boundries

In [33]:
maz_final_cleaned = "_15_maz_final_cleaned"
temp_small_polys = "temp_small_polys"

# 1. Create a feature layer from the output of your previous eliminate step
arcpy.management.MakeFeatureLayer(
    in_features=maz_cross_2, 
    out_layer=temp_small_polys
)

# 2. Select the stuck microzones under 12 square meters
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_small_polys,
    selection_type="NEW_SELECTION",
    where_clause="Shape_Area < 50"
)

# 3. Eliminate them into their neighboring microzones, blocking crossover with taz_lines
arcpy.management.Eliminate(
    in_features=temp_small_polys,
    out_feature_class=maz_final_cleaned,
    ex_features=taz_lines
)

# Optional: Print how many were cleaned up
cleaned_count = int(arcpy.management.GetCount(temp_small_polys)[0])
print(f"Eliminated {cleaned_count} small polygons.")

Eliminated 1329 small polygons.


### Rerun Crossover Check

Output `_17_true_crossover` is empty because there are no microzones that crossover TAZ boundries

In [37]:
maz_shrunk = "_16_maz_shrunk_check"
true_crossovers = "_17_true_crossovers"
temp_layer = "temp_shrunk_layer"

# 1. Shrink microzones by 1cm to drop false positives from shared edges
arcpy.analysis.Buffer(
    in_features=maz_final_cleaned,
    out_feature_class=maz_shrunk,
    buffer_distance_or_field="-0.01 Meters"
)

# 2. Select any shrunk zones that cross the TAZ boundary
arcpy.management.MakeFeatureLayer(maz_shrunk, temp_layer)
arcpy.management.SelectLayerByLocation(
    in_layer=temp_layer,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz
)

arcpy.management.CopyFeatures(temp_layer, true_crossovers)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_17_true_crossovers'>

### Final Cleaned Layer

**_15_Maz_Final Cleaned**

- No gaps
- No sliver polygons
- No microzones that crossover taz boundaries

### Summary stats for microzone counts per TAZ, county, and city

For each aggregation level, this summarizes the distribution of microzone counts across areas: number of areas, minimum, maximum, mean, and median microzones per area. Areas with a null grouping value are excluded and reported.

In [38]:
from collections import Counter
from statistics import mean, median

summary_fields = [
    ("TAZ", "TAZID"),
    ("County", "CO_NAME"),
    ("City", "CITY_NAME"),
]

for area_name, field_name in summary_fields:
    microzones_per_area = Counter()
    missing_area_count = 0

    with arcpy.da.SearchCursor(maz_final_cleaned, [field_name]) as rows:
        for (area_value,) in rows:
            if area_value is None:
                missing_area_count += 1
            else:
                microzones_per_area[area_value] += 1

    counts = list(microzones_per_area.values())
    if not counts:
        raise ValueError(f"No non-null {field_name} values found in {maz_final_cleaned}.")

    print(f"{area_name} ({field_name}) microzones per area:")
    print(f"  Area count: {len(counts)}")
    print(f"  Min: {min(counts)}")
    print(f"  Max: {max(counts)}")
    print(f"  Mean: {mean(counts):.2f}")
    print(f"  Median: {median(counts):.2f}")
    if missing_area_count:
        print(f"  Microzones skipped (null {field_name}): {missing_area_count}")

TAZ (TAZID) microzones per area:
  Area count: 3562
  Min: 1
  Max: 38
  Mean: 4.03
  Median: 4.00
County (CO_NAME) microzones per area:
  Area count: 5
  Min: 531
  Max: 5484
  Mean: 2872.40
  Median: 1942.00
City (CITY_NAME) microzones per area:
  Area count: 109
  Min: 1
  Max: 1464
  Mean: 131.76
  Median: 73.00
